In [26]:
import os
import random
from torchvision import datasets, transforms
from PIL import Image
from PIL import Image, ImageEnhance
import torch
from torch.utils.data import DataLoader
import torchvision.transforms.functional as F

In [40]:
# Aggressive augmentation pipeline for fundus images

fundus_transforms = transforms.Compose([
    transforms.RandomRotation(20),  # small rotation
    transforms.RandomResizedCrop(512, scale=(0.9, 1.0)),  # keep resolution ~512
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),

    # Color & lighting adjustments
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.02),

    # Slight blur/noise to simulate acquisition conditions
    transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 1.0)),

    transforms.ToTensor()
])

In [41]:
# Function to augment and save

def augment_and_save_images(input_dir, output_dir, num_augmentations=10):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    image_files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    for i, file_name in enumerate(image_files):
        img_path = os.path.join(input_dir, file_name)
        pil_img = Image.open(img_path).convert("RGB")

        for j in range(num_augmentations):
            aug_img = fundus_transforms(pil_img)      
            aug_img_pil = transforms.ToPILImage()(aug_img)

            # Extra step: slight sharpening to maintain clarity
            enhancer = ImageEnhance.Sharpness(aug_img_pil)
            aug_img_pil = enhancer.enhance(1.2)  # 1.0 = original, >1 = sharper

            # Save in PNG (lossless, keeps quality)
            save_path = os.path.join(output_dir, f"aug_{i}_{j}.png")
            aug_img_pil.save(save_path, format="PNG", quality=100)

# Example usage
input_dir = r"C:\Users\HP\Desktop\2311479\ADAM\ADAM\Test\Test-image-400\0"
output_dir = r"C:\Users\HP\Desktop\adamtest\0" 

augment_and_save_images(input_dir, output_dir, num_augmentations=15)